# Step 3: Optimizing Triton Attention Kernels

In this step, we take a hands-on approach to kernel optimization by writing and profiling custom attention kernels in [Triton](https://triton-lang.org/). Triton is a Python-based language and compiler for writing highly efficient GPU kernels, developed by OpenAI and widely used in production ML systems. Triton compiles Python to PTX/CUBIN with debug info, enabling full source correlation in the profiler.

```python
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    x = tl.load(x_ptr + offs, mask=offs < n)
    y = tl.load(y_ptr + offs, mask=offs < n)
    tl.store(out_ptr + offs, x + y, mask=offs < n)
```

We'll start with a naive baseline implementation and iteratively improve it, using Nsight Compute at each step to understand what changed and verify our optimizations had the expected effect. This workflow — profile, analyze, optimize, repeat — is the core of GPU performance engineering.

By the end of this step, you'll see how seemingly small changes to memory access patterns can yield dramatic speedups (20×+), and how the profiler guides you toward the right optimizations.

### What Does Prefill Attention Do?

In Step 2, we found that the prefill kernel in our previous script took a lot of time, compared to others. We want to profile a *custom* prefill attention kernel (since NVIDIA's pre-made, highly optimized kernels don't make for good optimization candidates).

To recap, during prefill, we compute attention over all input tokens at once:

$\displaystyle $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

- **Q** (Query): all input tokens asking "what should I attend to?"
- **K** (Key): all tokens answering "here's what I contain"
- **V** (Value): all tokens providing "here's my information"

The computation is dominated by two matrix multiplications:
1. **Score computation**: `Q @ K^T` → `[batch, seq_q, seq_kv]` — computes attention weights
2. **Weighted sum**: `softmax(scores) @ V` → `[batch, seq_q, head_dim]` — aggregates values

**The key optimization opportunity**: K and V tensors are the same for every query in the batch. If we're clever about how we load and reuse this data, we can improve out kernel.

---

## Baseline: `attention_prefill_dot`

Let's start with a deliberately naive implementation to establish a baseline. Our baseline kernel processes **one query per thread block**. This is simple to understand but wasteful:

Each block:
1. Loads its single query Q from global memory
2. Iterates over all K/V blocks, loading each independently from global memory
3. Computes attention and stores the result

**The problem**: With `seq_q=64` queries and `batch×heads=256`, we launch 16,384 thread blocks. Each block independently loads the *same* K/V data from global memory. There's no reuse — the same K/V tensors are loaded 64 times (once per query), wasting massive amounts of memory bandwidth.

This is a classic example of a memory-bound kernel that could be compute-bound with better data reuse. Let's profile it and see exactly what the numbers look like.

### Profile the Baseline

Execute the next cell to write the kernel source to disk. This allows us to use `--import-source yes` and have Triton/SASS correlation in the UI without manual resolve later on.

In [ ]:
%%writefile /dli/task/ncu/source/attention_prefill_dot.py
import torch
import triton
import triton.language as tl

@triton.jit
def attention_prefill_dot(
    Q_ptr, K_ptr, V_ptr, Out_ptr,
    stride_qb, stride_qm, stride_qk,
    stride_kb, stride_kn, stride_kk,
    stride_vb, stride_vn, stride_vk,
    stride_ob, stride_om, stride_ok,
    seq_q, seq_kv, scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    """Baseline: One query per block. No K/V reuse across queries."""
    batch_head = tl.program_id(0)
    query_idx = tl.program_id(1)

    if query_idx >= seq_q:
        return

    # Load single query [HEAD_DIM]
    q_ptrs = Q_ptr + batch_head * stride_qb + query_idx * stride_qm + tl.arange(0, HEAD_DIM)
    q = tl.load(q_ptrs)

    m_i = float('-inf')
    l_i = 0.0
    acc = tl.zeros([HEAD_DIM], dtype=tl.float32)

    # Iterate over KV (each block loads K/V independently!)
    for n_start in range(0, seq_kv, BLOCK_N):
        n_offs = n_start + tl.arange(0, BLOCK_N)
        n_mask = n_offs < seq_kv

        # Load K [BLOCK_N, HEAD_DIM]
        k_ptrs = K_ptr + batch_head * stride_kb + n_offs[:, None] * stride_kn + tl.arange(0, HEAD_DIM)[None, :]
        k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)

        # Q @ K^T
        k_t = tl.trans(k)
        qk_2d = tl.dot(q[None, :], k_t.to(tl.float16))
        qk = tl.sum(qk_2d, axis=0) * scale

        # Online softmax
        m_new = tl.maximum(m_i, tl.max(qk))
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(qk - m_new)
        p = tl.where(n_mask, p, 0.0)
        l_new = alpha * l_i + tl.sum(p)

        # Load V and accumulate
        v_ptrs = V_ptr + batch_head * stride_vb + n_offs[:, None] * stride_vn + tl.arange(0, HEAD_DIM)[None, :]
        v = tl.load(v_ptrs, mask=n_mask[:, None], other=0.0)
        pv_2d = tl.dot(p[None, :].to(tl.float16), v)
        acc = alpha * acc + tl.sum(pv_2d, axis=0)
        m_i, l_i = m_new, l_new

    # Store
    acc = acc / l_i
    o_ptrs = Out_ptr + batch_head * stride_ob + query_idx * stride_om + tl.arange(0, HEAD_DIM)
    tl.store(o_ptrs, acc.to(tl.float16))

torch.manual_seed(42)
B, H, SQ, SKV, D = 8, 32, 64, 4096, 64  # seq_kv=4096 gives ~16GB DRAM loads
q = torch.randn(B*H, SQ, D, dtype=torch.float16, device='cuda')
k = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda')
v = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda')
out = torch.empty_like(q)
attention_prefill_dot[(B*H, SQ)](q, k, v, out, *q.stride(), *k.stride(), *v.stride(), *out.stride(), SQ, SKV, 1.0/(D**0.5), BLOCK_M=1, BLOCK_N=64, HEAD_DIM=D)
torch.cuda.synchronize()

---

### Profile and Analyze the Kernel

In [ ]:
!ncu --set full --import-source yes -k regex:attention \
    -f -o /dli/task/ncu/reports/attention_prefill_dot \
    python3 source/attention_prefill_dot.py

Switch to the Nsight Streamer tab, or re-open it by selecting **NVIDIA Nsight** > **Display Nsight UI via Nsight Streamer** if it's closed.

<img src="images/jupyterlab_menu_2.png"/>

Open the report `/home/nvidia/reports/attention_prefill_dot.ncu-rep`.

The *GPU Speed of Light* section confirms that we see very high memory utilization: a memory-bound kernel.

<table style="margin-left: 0;">
    <tr><th>Metric</th><th>Observation</th><th>Interpretation</th></tr>
    <tr><td><b>Duration</b></td><td>~5 ms</td><td>Our baseline to beat</td></tr>
    <tr><td><b>Grid Size</b></td><td>16,384</td><td>Many small blocks, each doing redundant work</td></tr>
    <tr><td><b>Memory Throughput</b></td><td>~85%</td><td>We're heavily utilizing DRAM bandwidth — the memory system is the bottleneck</td></tr>
    <tr><td><b>Compute Throughput</b></td><td>~50%</td><td>Compute units are underutilized, waiting for data</td></tr>
</table>

The *Warp State Statistics* section shows long and short scoreboard stalls — warps (groups of threads) waiting for memory loads.

We can switch to the *Source page* and use Nsight Compute's new **Instructions & Scoreboards** [table](https://docs.nvidia.com/nsight-compute/NsightCompute/index.html#instructions-scoreboards-table) to identify the source lines causing these stalls.

<img src="images/ncu_ui_3_sb.png" width="50%"/>

This table shows three sets of information for the selected line:

- Self Instruction Mix: The breakdown of all instruction categories. Use this to understand the composition of the selection, and which instruction categories have the most (scoreboard) warp stalls.

- Input Scoreboard Dependencies: The scoreboard-driven data dependencies of the selection. Data dependencies are predecessor operations that need to complete before code in the selection can proceed. Use this to understand on which instructions the selection is waiting (stalled) on. The Attributed Stalls column shows the number of warp stalls from the selection attributed to this dependency. As a result, dependencies causing more stalls in the selection are sorted higher in the table. This data is also available in the source views to locate lines causing the most stalls.

- Output Scoreboard Dependencies: The consumers of scoreboards produced by the selection. A scoreboard producer is an operation producing a scoreboard a consumer is waiting on. Each line waited on the selection and potentially other instructions. Use this to understand who is waiting (stalled) on the selection. Note that these lines may wait for other instructions than the current selection, too.

**The key insight from profiling**: We're loading the same K/V data 64 times (once per query). If we process multiple queries per block, we can load K/V once and reuse it for all queries in that block. This is the fundamental idea behind tiled attention kernels like Flash Attention.

<img src="images/ncu_ui_3_dot.png" width="75%"/>

---

## Optimization 1: `attention_prefill_fp8`

Before we try algorithmic changes, let's see what happens if we just reduce the data size to reduce memory pressure. The H100 supports FP8 (8-bit floating point), which halves the memory traffic compared to FP16.

**Key Change**: Use FP8 inputs instead of FP16, but keep the same "one query per block" structure as the baseline.

<table style="margin-left: 0;">
    <tr><th>Precision</th><th>Bytes/Element</th><th>Bandwidth Savings</th></tr>
    <tr><td>FP16 (baseline)</td><td>2</td><td>—</td></tr>
    <tr><td>FP8 (E4M3)</td><td>1</td><td><b>2× reduction</b></td></tr>
</table>

This is a simple optimization that doesn't require any algorithmic changes — we're just using a smaller data type. Let's see how much speedup we get from bandwidth reduction alone, before adding tiling.

In [ ]:
%%writefile /dli/task/ncu/source/attention_prefill_fp8.py
import torch
import triton
import triton.language as tl

@triton.jit
def attention_prefill_fp8(
    Q_ptr, K_ptr, V_ptr, Out_ptr,
    stride_qb, stride_qm, stride_qk,
    stride_kb, stride_kn, stride_kk,
    stride_vb, stride_vn, stride_vk,
    stride_ob, stride_om, stride_ok,
    seq_q, seq_kv, scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    """FP8 Baseline: One query per block with FP8 inputs for 2× bandwidth reduction."""
    batch_head = tl.program_id(0)
    query_idx = tl.program_id(1)

    if query_idx >= seq_q:
        return

    # Load single query as FP8 [HEAD_DIM]
    q_ptrs = Q_ptr + batch_head * stride_qb + query_idx * stride_qm + tl.arange(0, HEAD_DIM)
    q = tl.load(q_ptrs)

    m_i = float('-inf')
    l_i = 0.0
    acc = tl.zeros([HEAD_DIM], dtype=tl.float32)

    # Iterate over KV (each block loads K/V independently — still no reuse!)
    for n_start in range(0, seq_kv, BLOCK_N):
        n_offs = n_start + tl.arange(0, BLOCK_N)
        n_mask = n_offs < seq_kv

        # Load K as FP8 [BLOCK_N, HEAD_DIM]
        k_ptrs = K_ptr + batch_head * stride_kb + n_offs[:, None] * stride_kn + tl.arange(0, HEAD_DIM)[None, :]
        k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)

        # FP8 × FP8 -> FP32 accumulation
        k_t = tl.trans(k)
        qk_2d = tl.dot(q[None, :].to(tl.float16), k_t.to(tl.float16))
        qk = tl.sum(qk_2d, axis=0) * scale

        # Online softmax in FP32
        m_new = tl.maximum(m_i, tl.max(qk))
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(qk - m_new)
        p = tl.where(n_mask, p, 0.0)
        l_new = alpha * l_i + tl.sum(p)

        # Load V as FP8 and accumulate
        v_ptrs = V_ptr + batch_head * stride_vb + n_offs[:, None] * stride_vn + tl.arange(0, HEAD_DIM)[None, :]
        v = tl.load(v_ptrs, mask=n_mask[:, None], other=0.0)
        pv_2d = tl.dot(p[None, :].to(tl.float16), v.to(tl.float16))
        acc = alpha * acc + tl.sum(pv_2d, axis=0)
        m_i, l_i = m_new, l_new

    # Store as FP16
    acc = acc / l_i
    o_ptrs = Out_ptr + batch_head * stride_ob + query_idx * stride_om + tl.arange(0, HEAD_DIM)
    tl.store(o_ptrs, acc.to(tl.float16))

torch.manual_seed(42)
B, H, SQ, SKV, D = 8, 32, 64, 4096, 64  # seq_kv=4096 gives ~16GB DRAM loads
# Pre-quantize to FP8
q = torch.randn(B*H, SQ, D, dtype=torch.float16, device='cuda').to(torch.float8_e4m3fn)
k = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda').to(torch.float8_e4m3fn)
v = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda').to(torch.float8_e4m3fn)
out = torch.empty(B*H, SQ, D, dtype=torch.float16, device='cuda')
attention_prefill_fp8[(B*H, SQ)](q, k, v, out, *q.stride(), *k.stride(), *v.stride(), *out.stride(), SQ, SKV, 1.0/(D**0.5), BLOCK_M=1, BLOCK_N=64, HEAD_DIM=D)
torch.cuda.synchronize()

---

### Profile and Compare the FP8 Kernel

In [ ]:
!ncu --set full --import-source yes -k regex:attention \
    -f -o /dli/task/ncu/reports/attention_prefill_fp8 \
    python3 source/attention_prefill_fp8.py

One of Nsight Compute's most powerful features is comparing results using baselines. When you set a result as a baseline, all other results show metrics relative to it. This makes it easy to quantify the impact of optimizations and verify that changes had the expected effect.

<img src="images/ncu_ui_1_baselines.png" width="50%"/>

<div class="alert alert-block alert-info">
    <p>Exercise: Compare FP8 vs Baseline</p>
</div>

We can use the profiler's baseline comparison feature to quantify exactly what changed. This is one of Nsight Compute's most powerful features for iterative optimization.
Compare this report with the FP16 baseline:

1. Open/select `/home/nvidia/reports/attention_prefill_dot.ncu-rep` in the UI
2. Click **Compare > Add Baseline** to set it as the baseline
3. Open `/home/nvidia/reports/attention_prefill_fp8.ncu-rep` — all values now show relative differences

<img src="images/add_baseline.png"/>

<table style="margin-left: 0;">
    <tr><th>Metric</th><th>Value</th><th>Change</th></tr>
    <tr><td><b>Duration</b></td><td>~3.3 ms</td><td><b>~1.5× faster</b> — improvement from traffic reduction alone</td></tr>
    <tr><td><b>Memory Throughput</b></td><td>~70%</td><td>Lower, since only moving half the memory size</td></tr>
    <tr><td><b>Compute Throughput</b></td><td>~65%</td><td>More balanced — compute catching up</td></tr>
    <tr><td><b>Grid Size</b></td><td>16,384</td><td>Same — still one query per block</td></tr>
</table>

**Key insight**: FP8 gives us a nice speedup (~1.5×) by halving memory traffic. The kernel is now more compute-memory balanced, but we're still loading K/V redundantly for each query.

**This sets up a natural question**: What if we combine FP8 (smaller data) with tiling (less redundant loads)? The profiler shows we're still launching 16,384 blocks, each loading the same K/V data independently. Let's fix that next.

---

## Optimization 2: `attention_prefill_tiled_fp8`

We reduced each element from 2 to 1 bytes, but we are still loading K/V redundantly for every query. We can use **tiling**  to process multiple queries per block and thereby reuse the K/V cache. This is on top of the FP8 data size and bandwidth reduction.

**Key Change**: Process `BLOCK_M=16` queries per thread block instead of 1

<table style="margin-left: 0;">
    <tr><th>Aspect</th><th>FP8 Baseline</th><th>Tiled FP8</th></tr>
    <tr><td>Queries per block</td><td>1</td><td>BLOCK_M (16)</td></tr>
    <tr><td>Grid size</td><td>batch×heads×seq_q = 16,384</td><td>batch×heads×(seq_q/BLOCK_M) = 1,024</td></tr>
    <tr><td>K/V loads</td><td>Redundant — loaded once per query</td><td>Shared — loaded once per 16 queries</td></tr>
    <tr><td>Element size</td><td>1 byte (FP8)</td><td>1 byte (FP8)</td></tr>
</table>

The math is simple but powerful: each K/V block is now loaded once and reused for all 16 queries in the tile. This reduces memory traffic by 16×.

This is exactly the same principle behind Flash Attention and other optimized attention implementations — they use tiling for data reuse, often combined with lower precision for bandwidth efficiency.

In [ ]:
%%writefile /dli/task/ncu/source/attention_prefill_tiled_fp8.py
import torch
import triton
import triton.language as tl

@triton.jit
def attention_prefill_tiled_fp8(
    Q_ptr, K_ptr, V_ptr, Out_ptr,
    stride_qb, stride_qm, stride_qk,
    stride_kb, stride_kn, stride_kk,
    stride_vb, stride_vn, stride_vk,
    stride_ob, stride_om, stride_ok,
    seq_q, seq_kv, scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    """FP8 Tiled: Native FP8 inputs for 2× bandwidth reduction."""
    batch_head = tl.program_id(0)
    m_block_idx = tl.program_id(1)

    m_offs = m_block_idx * BLOCK_M + tl.arange(0, BLOCK_M)
    m_mask = m_offs < seq_q

    # Load Q as FP8 (1 byte per element!)
    q_ptrs = Q_ptr + batch_head * stride_qb + m_offs[:, None] * stride_qm + tl.arange(0, HEAD_DIM)[None, :]
    q = tl.load(q_ptrs, mask=m_mask[:, None], other=0.0)

    m_i = tl.full([BLOCK_M], float('-inf'), dtype=tl.float32)
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
    acc = tl.zeros([BLOCK_M, HEAD_DIM], dtype=tl.float32)

    for n_start in range(0, seq_kv, BLOCK_N):
        n_offs = n_start + tl.arange(0, BLOCK_N)
        n_mask = n_offs < seq_kv

        # Load K as FP8
        k_ptrs = K_ptr + batch_head * stride_kb + n_offs[:, None] * stride_kn + tl.arange(0, HEAD_DIM)[None, :]
        k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)

        # FP8 × FP8 -> FP32 accumulation
        qk = tl.dot(q, tl.trans(k)) * scale
        qk = tl.where(n_mask[None, :], qk, float('-inf'))

        # Softmax in FP32 (critical for accuracy!)
        m_ij = tl.max(qk, axis=1)
        m_new = tl.maximum(m_i, m_ij)
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(qk - m_new[:, None])
        l_new = alpha * l_i + tl.sum(p, axis=1)

        # Load V as FP8
        v_ptrs = V_ptr + batch_head * stride_vb + n_offs[:, None] * stride_vn + tl.arange(0, HEAD_DIM)[None, :]
        v = tl.load(v_ptrs, mask=n_mask[:, None], other=0.0)

        # P @ V: FP16 for P (precision), FP8 for V
        acc = alpha[:, None] * acc + tl.dot(p.to(tl.float16), v.to(tl.float16))
        m_i, l_i = m_new, l_new

    # Output as FP16
    acc = acc / l_i[:, None]
    o_ptrs = Out_ptr + batch_head * stride_ob + m_offs[:, None] * stride_om + tl.arange(0, HEAD_DIM)[None, :]
    tl.store(o_ptrs, acc.to(tl.float16), mask=m_mask[:, None])

torch.manual_seed(42)
B, H, SQ, SKV, D, BM = 8, 32, 64, 4096, 64, 16  # seq_kv=4096 gives ~16GB DRAM loads
# Pre-quantize to FP8
q = torch.randn(B*H, SQ, D, dtype=torch.float16, device='cuda').to(torch.float8_e4m3fn)
k = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda').to(torch.float8_e4m3fn)
v = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda').to(torch.float8_e4m3fn)
out = torch.empty(B*H, SQ, D, dtype=torch.float16, device='cuda')
attention_prefill_tiled_fp8[(B*H, (SQ+BM-1)//BM)](q, k, v, out, *q.stride(), *k.stride(), *v.stride(), *out.stride(), SQ, SKV, 1.0/(D**0.5), BLOCK_M=BM, BLOCK_N=64, HEAD_DIM=D)
torch.cuda.synchronize()

---

### Profile and Compare the Tiled Kernel

In [ ]:
!ncu --set full --import-source yes -k regex:attention \
    -f -o /dli/task/ncu/reports/attention_prefill_tiled_fp8 \
    python3 source/attention_prefill_tiled_fp8.py

<div class="alert alert-block alert-info">
    <p>Exercise: Compare FP8 Tiled vs FP8 Baseline</p>
</div>

Again, let's use the profiler's baseline comparison feature to quantify exactly what changed. 

1. Select **Compare > Clear Baselines** to remove the original baseline report - we only want to compare versions 2 and 3 now
2. Open/select `/home/nvidia/reports/attention_prefill_fp8.ncu-rep`
3. Click **Compare > Add Baseline** to set it as the new baseline
4. Open `/home/nvidia/reports/attention_prefill_tiled_fp8.ncu-rep` — all values now show relative differences between these two

<img src="images/clear_baselines.png"/>

<table style="margin-left: 0;">
    <tr><th>Metric</th><th>Value</th><th>Change vs FP8</th></tr>
    <tr><td><b>Duration</b></td><td>~0.24 ms</td><td><b>~14× faster</b></td></tr>
    <tr><td><b>Grid Size</b></td><td>1,024</td><td>16× fewer blocks</td></tr>
    <tr><td><b>Memory Throughput</b></td><td>~70%</td><td>Similar — but doing far less redundant work</td></tr>
    <tr><td><b>Compute Throughput</b></td><td>~60%</td><td>Slightly lower — kernel is now very short</td></tr>
</table>

**Why it's faster**: By processing 16 queries per block, each K/V load is reused 16×. This transforms the kernel from doing massive redundant loads to efficiently reusing data. The profiler confirms our hypothesis — we reduced memory traffic and the speedup matches our expectations.

We're still computing the same attention values. We just reorganized *how* we load the data to reduce traffic and maximize reuse.

---

## Final Comparison

Let's step back and see what we accomplished:

<table style="margin-left: 0;">
    <tr><th>Kernel</th><th>Duration</th><th>Speedup vs Baseline</th><th>Key Change</th></tr>
    <tr><td><code>attention_prefill_dot</code></td><td>5 ms</td><td>1× (baseline)</td><td>Naive: one query per block, FP16</td></tr>
    <tr><td><code>attention_prefill_fp8</code></td><td>3.3 ms</td><td><b>~1.5×</b></td><td>FP8 inputs (half bandwidth)</td></tr>
    <tr><td><code>attention_prefill_tiled_fp8</code></td><td>0.24 ms</td><td><b>~20×</b></td><td>Tiling + FP8 (16 queries per block, K/V reuse)</td></tr>
</table>

Every optimization was *guided by the profiler*. We didn't guess — we measured, identified the bottleneck, applied a targeted fix, and verified the improvement. This is the systematic approach to GPU optimization.

<img src="images/ncu_ui_3_fp8_tiled_chart.png" width="75%"/>

### Summary

<div class="alert alert-block alert-success">
    <p><b>FP8 Quantization</b>: Use 8-bit floats for 2× memory bandwidth reduction. Simple change, ~1.5× speedup.</p>
    <p><b>Tiling</b>: Process multiple queries per block to reuse K/V loads. Reduced redundant loads by 16×.</p>
    <p><b>Profiler guidance</b>: High Memory Throughput + Low(er) Compute Throughput → memory-bound. Look for ways to reduce loads or increase reuse.</p>
    <p><b>Baselines</b>: Compare reports to quantify the impact of each optimization.</p>
</div>

**Next: [Step 4 - Nsight Python](step4.ipynb)**